# Get lookthrough holdings

In [1]:
print('#######################')
print('#  START lt_dl.ipynb  #')
print('#######################')

#######################
#  START lt_dl.ipynb  #
#######################


In [2]:
import time
start_time       = time.time()
start_time_lt_dl = time.time()

num_batches = 17

# libraries, libraries!
print("Importing libraries ...")
from datetime import datetime
import pandas as pd
import os, re
from pathlib import Path
from tqdm import tqdm
from constants import pthPy, pth_dl, pthTest
from utilities import (
    timediff,
    last_working_day,
    prior_month_end,
    osprey,
    batch_list,
    r_classifier,
)
import subprocess

# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException, TimeoutException, UnexpectedAlertPresentException

print(f' {timediff(start_time, time.time())} importing libraries\n')

Importing libraries ...
 16.9sec importing libraries



In [3]:
# get inputs to pass to Eagle
start_time = time.time()
print('Collecting input data ...')

# get fund codes from r28_cs1 tab of the py_report.xlsm sheet
df      = pd.read_excel(pthPy, sheet_name="downloader", usecols="A,E").dropna(subset = ['Fund'])
funds   = df['Fund'].apply(str.upper)

# get report date
k       = df.iloc[0,1]
rptDate = k if isinstance(k, datetime) else prior_month_end(datetime.today()) # prior month end or report date override; type is datetime()

# check inputs
s = "" if len(funds) == 1 else "s"
print(f'{len(funds)} lookthrough{s} as at {rptDate.strftime("%A %d %b %Y")} to be downloaded:\n  {(",").join(funds)}')
print(f'\n{timediff(start_time, time.time())} collecting input data\n')

142 lookthroughs as at Friday 31 Jul 2026 to be downloaded:
  3BBCIINC,ABMMBND,ADRRC,ADVMB,AMMARF,AMPBQP,ASBTOS,ASHFLX,BCIFIF,BPROV,CCNPF,CMPFFLEX,CMPFINC,CSIRBQP,ECICBALC,ELCIPF,ENGENIP,ENGENMBF,FEMPBF,GAEMBF,GEMSMEDC,GMRETF,GMRETF2,GRFINV,GTCWP2,HOLADC,HOLDINC,HOLIDC,HOLYPF,HOSMED,IJGCOR,IJGIPF,IMPALA,IMPBAL_C,IMPREF_C,IPIPF,ISPFP,LEZAFFI,LEZALDI,LPIIFTAA,MASAINC,MASAINRF,MASAMMF,MASASI,MASASIRF,MEDINC,MOMBBF,MOMBBRF,MOMFLX,MOMFXB,MOMIPF,MOMPRET,MOMTAAHI,MOMTAALI,MOMTAAMI,MULTICH,MWPFEQU,MWPFILB,MYQIP,NESEQU,NFMWAGG,NGKINC,OMMAIF,PABS,PBNDQ,PBFETF,PCBF,PCCEF,PCEQTF,PCGEARF,PCGEF,PCSHQ,PEQ,PEQF,PETFIP,PEYF,PFFIF,PGCBF,PGCEF,PGIPFA,PGPCEM_C,PGPCGE_C,PGPCZAR,PGPGARF,PGPGBF_C,PGPGIF_C,PGPRF_C,PICPROV,PIF,PIMBAL,PIMEVO,PIMIDF,PIPF,PIPFP,PLMED,PLPRNA,PMMF,POIF_C,PORFIP,PPEF,PPOS,PPSBAL,PPSBAL_C,PPSFLEX,PPSNAM,PPSRBNQ,PPWEQU,PRPABF,PRPDBF,PSIF,PSILB,PSPAM,PSSPFEQU,PSTIF,PTIF,QIFFGIF,SAAMCAU,SAAMINC,SAAMMOD,SABCCSHF,SCBPF,SDINCPM,SILAIF,SILEQF,SISLP,SMMAIF,SMMIBF,SMMILP,SMMRRF,SNPFEQU,SSLSP,

In [4]:
# download the lookthrough reports in batches
start_time = time.time()
suffix = "csv"
batch_size = int(len(funds) / num_batches)
batches = batch_list(funds, batch_size=min(len(funds), max(batch_size, 1)))
print(f"Downloading the {len(funds)} lookthroughs \
    for {rptDate.strftime('%a %d %b %Y')} in {num_batches} batches ...")

batch_filepaths = []
for index, batch in tqdm(enumerate(batches, start=1)):
    # give status of downloads
    pattern = re.compile(
        rf"^R28I.*of_{len(batches)}.*{rptDate.strftime('%d%b%Y')}\.{suffix}$"
    )
    matches = [f.name for f in Path(pth_dl).iterdir() if f.is_file() and pattern.match(f.name)]
    done = [
        re.search(r"R28I\s+(\d+)_of", f).group(1)
        for f in matches
        if re.search(r"R28I\s+(\d+)_of", f)
    ]
    es = "es" if len(batches) - len(done) != 1 else ""
    print(f"\n{(', ').join(list(done))} out of {len(batches)} batches done")
    print(f"{len(batches) - len(done)} batch{es} to go\n")

    # create the name of the file to be downloaded
    fln = f"{index}_of_{len(batches)}_LT"
    filename = f"R28I {fln}({len(batch)}) {rptDate.strftime('%#d%b%Y')}.{suffix}"
    batch_filepath = os.path.join(pth_dl, filename)
    batch_filepaths.append(batch_filepath)
    print(f"Get {filename}, a batch of {len(batch)} files:\n   {(', ').join(batch)}")

    if os.path.isfile(batch_filepath):
        print(f"\n{batch_filepath} exists\n")
        pass
    else:
        try:
            # print(f"Get {filename}, a batch of {len(batch)} files:\n   {(', ').join(batch)}")
            start_time_1 = time.time()
            print(f"Downloading batch {index} of {len(batches)} as {batch_filepath} ...")
            osprey("r28i", (",").join(batch), rptDate, rptDate, fln, suffix)
            print(f" {timediff(start_time_1, time.time())} to download batch {index} of {len(batches)}\n")
        except:
            pass

undone_num = [i for i in range(1, len(batches) + 1) if i not in list(map(int, done))]
undone_grp = [list(batch) for i, batch in enumerate(batches, start=1) if i in undone_num]
undone_lst = [item for sublist in undone_grp for item in sublist]

if len(undone_lst) == 0:
    print(
        f"\n{timediff(start_time, time.time())} downloading \
the {len(funds)} lookthroughs for {rptDate.strftime('%a %d %b %Y')} \
in {num_batches} batches\n"
    )
    print("#######################################")
    print("#    DONE DOWNLOADING LOOKTHROUGHS    #")
    print("#######################################\n")
else:
    raise Exception as e
        print(f"\n{len(undone_num)} batch{es}, {len(undone_lst)} \
        funds, not downloaded:\n {(', ').join(undone_lst)}\n")

        except Exception as e:
            print(e)

SyntaxError: invalid syntax (2657903835.py, line 59)

In [ ]:
print(
        f"\n{len(undone_num)} batch{es}, {len(undone_lst)} funds, not downloaded:\n {(',').join(undone_lst)}\n"
    )

In [ ]:
undone = [i for i in range(1, len(batches) + 1) if i not in list(map(int, done))]
undone_list = [list(batch) for i, batch in enumerate(batches, start=1) if i in undone]
undone_list

In [ ]:
print(
        f"{len(undone)} batch{es} not downloaded: {(',').join(undone_lst)}"
    )

In [ ]:
undone = []
for i in range(1, len(batches) + 1):
    if i not in list(map(int, done)):
        undone.append(i)

undone

In [ ]:
undone = [i for i in range(1, len(batches) + 1) if i not in list(map(int, done))]
undone

In [ ]:
done

In [ ]:
int('8') not in range(1, len(batches) + 1)

In [ ]:
print(range(1, len(batches) + 1))

In [ ]:
my_range = range(5, 10)
my_range

In [ ]:
8 in range(1, len(batches) + 1)

In [ ]:
# dataframe the lookthrough holdings
start_time = time.time()
print("Dataframing the lookthroughs\n")

holdings = pd.DataFrame()
for batch_filepath in batch_filepaths:
    # print(f" {batch_filepath}")
    df_new = pd.read_csv(batch_filepath)
    holdings = pd.concat([holdings, df_new])

In [ ]:
holdings

In [ ]:
holdings.info()

In [ ]:
# convert date columns from type object
date_cols = ["Next Coupon Date", "Maturity Date", "i Position Effective Date"]
for date_col in date_cols:
    holdings[date_col] = pd.to_datetime(holdings[date_col])

# convert value columns from type object to type float
value_cols = [
    "Original Nominal",
    "Clean Book Value",
    "Clean Market Value",
    "Accrued Income",
    "Dividend Receivable",
    "Sum of Market Value Income",
    "Market Value %",
    "Current Exposure",
]
for value_col in value_cols:
    holdings[value_col] = holdings[value_col].str.replace(",", "").astype(float)

print(
    f" {len(holdings['Entity Name'].unique())} fund holdings \
as at {holdings['i Position Effective Date'].iloc[0].strftime('%d %b %Y')} in the dataframe"
)

print(f"\n{timediff(start_time, time.time())} dataframing the lookthroughs\n")

In [ ]:
# get the fund NAVs
start_time = time.time()
print(
    f"Getting the {len(funds)} lookthrough funds' NAVs as at {rptDate.strftime('%A %d %B %Y')} with osprey() ..."
)

name = "LT"
navs_fln = os.path.join(
    pth_dl,
    f"FNAV {name}({len(funds)}) {rptDate.strftime('%d%b%Y')}.csv",
)

if os.path.exists(navs_fln):
    print(
        f" Lookthrough fund NAVs as at {rptDate.strftime('%a %d %b %Y')} already downloaded: {navs_fln}"
    )
    pass
else:
    osprey("fnav", (",").join(funds), rptDate, rptDate, name, "csv")

# dataframe the downloaded fund NAVs
navs = pd.read_csv(navs_fln)

# convert the Total column from object to float
navs["Total Net Assets"] = (
    navs["Total Net Assets"].str.replace(",", "").astype("float64")
)
# navs['Total Net Assets'] = navs['Total Net Assets'].apply(lambda x: f"{x:,.2f}") # present with thousands separator and to two decimals

print("\n", navs_fln)

print(
    f" {timediff(start_time, time.time())} getting the {len(funds)} funds' NAV{'s' if len(funds) != 1 else ''} as at {rptDate.strftime('%A %d %B %Y')} \
with osprey()"
)

In [ ]:
holdings

In [ ]:
navs

In [ ]:
holdings_totals = holdings.groupby('Entity Name', as_index = False).sum(numeric_only = True)[['Entity Name','End Market Value','Closing Exposure PA']]
holdings_totals

In [ ]:
holdings_totals = holdings.groupby('Entity Name', as_index = False).sum(numeric_only = True)[['Entity Name','End Market Value','Closing Exposure PA']]
sums_cf = holdings_totals.merge(navs, how = 'left', left_on = 'Entity Name', right_on = 'NAV Entity ID', suffixes = (None, '_y'))
sums_cf.drop(['Entity Name_y', 'NAV Entity ID'], axis = 1, inplace = True)
sums_cf['EMV-CEPA'] = sums_cf['End Market Value'] - sums_cf['Closing Exposure PA']
sums_cf['EMV-TNA']  = sums_cf['End Market Value'] - sums_cf['Total Net Assets']
sums_cf = sums_cf.sort_values(by='EMV-TNA', ascending=False)
sums_cf

In [ ]:
sums_cf['EMV-CEPA']  = sums_cf['End Market Value'] - sums_cf['Closing Exposure PA']
sums_cf['EMV-TNA']   = sums_cf['End Market Value'] - sums_cf['Total Net Assets']

In [ ]:
# merge the lookthrough holdings and NAVs, and compare their totals
start_time = time.time()
print(
    f"Merging and comparing the {len(funds)} lookthroughs and NAVs as at {rptDate.strftime('%A %d %B %Y')} ..."
)

holdings_totals = holdings.groupby("Entity ID", as_index=False).sum(numeric_only=True)[
    ["Entity ID", "Sum of Market Value Income", "Current Exposure"]
]
sums_cf = holdings_totals.merge(
    navs, how="left", left_on="Entity ID", right_on="NAV Entity ID"
)
sums_cf.drop(["Entity Name", "NAV Entity ID"], axis=1, inplace=True)
sums_cf["SoMVI-CE"] = (
    sums_cf["Sum of Market Value Income"] - sums_cf["Current Exposure"]
)
sums_cf["1-CE/SoMVI %"] = (
    1 - sums_cf["Current Exposure"] / sums_cf["Sum of Market Value Income"]
) * 100
sums_cf["SoMVI-NAV"] = abs(
    sums_cf["Sum of Market Value Income"] - sums_cf["Total Net Assets"]
)
sums_cf["1-NAV/SoMVI %"] = (
    1 - sums_cf["Total Net Assets"] / sums_cf["Sum of Market Value Income"]
) * 100
sums_cf = sums_cf.sort_values(by="SoMVI-NAV", ascending=False)
cols_order = [
    "Effective Date",
    "Entity ID",
    "Sum of Market Value Income",
    "Current Exposure",
    "Total Net Assets",
    "SoMVI-CE",
    "1-CE/SoMVI %",
    "SoMVI-NAV",
    "1-NAV/SoMVI %",
]
sums_cf = sums_cf[cols_order]

# set the number of decimals to present
cols_2dp = ["Sum of Market Value Income", "Current Exposure", "Total Net Assets"]
for col in cols_2dp:
    sums_cf[col] = sums_cf[col].apply(lambda x: f"{x:,.2f}")

cols_6dp = ["SoMVI-CE", "1-CE/SoMVI %", "SoMVI-NAV", "1-NAV/SoMVI %"]
for col in cols_6dp:
    sums_cf[col] = sums_cf[col].apply(lambda x: f"{x:,.6f}")

# sums_cf
# ...

print(
    f" {timediff(start_time, time.time())} merging and comparing the {len(funds)} \
lookthroughs and NAVs as at {rptDate.strftime('%A %d %B %Y')}"
)

In [ ]:
# convert the lookthrough holdings into Reg 28 format with correspodning headings
start_time = time.time()
print(f"Converting the lookthrough holdings in readiness for Reg 28 classification ...")

s = "" if len(funds) == 1 else "s"
lt_fname = os.path.join(
    pthTest, f"LT holdings ({len(funds)}) {rptDate.strftime('%d%b%Y')}.xlsx"
)

hold_cols = [
    "Entity ID",
    "Investment Type",
    "i Issue Name",
    "PrimaryAssetID",
    "CCY",
    "Sum of Market Value Income",
    "% of Total Market Value",
    "Current Exposure",
]
hReg28 = holdings[hold_cols]  # identify the subset of holdings columns to be used
hReg28 = hReg28.rename(
    columns={
        "Entity ID": "Entity Name",
        "PrimaryAssetID": "Primary Asset ID",
        "Sum of Market Value Income": "End Market Value",
        "% of Total Market Value": "Percentage of Market Value",
        "Current Exposure": "Closing Exposure PA",
    }
)
hReg28.insert(
    5, "Reg28 Classification", ""
)  # insert the classification column as the new column 5
hReg28.insert(
    9, f"{rptDate.strftime('%d %b %Y')}", ""
)  # insert the report date as a header in the last column
hReg28.iloc[0, 9] = lt_fname
hReg28.iloc[1, 9] = "LT"
hReg28.reset_index(drop=True, inplace=True)

# hReg28.info()

print(
    f" {timediff(start_time, time.time())} converting the lookthrough holdings in readiness for Reg 28 classification"
)

In [ ]:
# write the lookthrough holdings dataframe to review it as a worksheet
start_time = time.time()
print("Writing the lookthrough holdings dataframe and navs dataframe to a sheet ...")

writer = pd.ExcelWriter(lt_fname, engine="xlsxwriter")  # instantiate a sheet writer
hReg28.to_excel(writer, index=False, sheet_name="All")  # write the NAV sheet
holdings.to_excel(writer, index=False, sheet_name="PARN")  # write the NAV sheet
sums_cf.to_excel(writer, index=False, sheet_name="NAVs")  # write the missing NAVs sheet
writer.close()  # https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html   class for writing DataFrame objects into excel sheets
print(f" \n{lt_fname}\n")

print(
    f" {timediff(start_time, time.time())} writing the lookthrough holdings dataframe and navs dataframe to a sheet\n"
)

In [ ]:
# run the Reg 28 reporting script
start_time = time.time()
print(f"Classifying the lookthrough holdings\n")

r_classifier("lt", lt_fname)

print(f" \n{timediff(start_time, time.time())} classifying the lookthrough holdings\n")

# # run the Reg28 reporting script
# start_time = time.time()
# print(f"Generating the lookthrough reports\n")

# subprocess.run(
#     ["python", "C:/Users/hilton.netta/OneDrive - Prescient/py/gitrepo/issuers_1.py"]
# )

print(f" \n{timediff(start_time, time.time())} generating the lookthrough reports\n")


In [ ]:

print(
    "\n",
    f"{timediff(start_time_lt_dl, time.time())} roundtrip to download and merge lookthroughs\n",
)

print("\n\n#####################")
print("#    END lt_dl.py   #")
print("#####################")